In [ ]:
import os
import json
import time
import re
import logging
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import pypdf


GROQ_API_KEY = ""   #
DATA_FOLDER = "data"                    # Dossier contenant les PDFs
OUTPUT_FILE = "data/qa_finetuning.jsonl"
FICHIER_BRUT = "data/qa_brut.jsonl"
FICHIER_PROGRESSION = "data/progression.json"  # Pour reprendre si interruption

QUESTIONS_PAR_CHUNK = 5    # Nombre de QA par chunk
CHUNK_SIZE = 800            # Taille des chunks en caractères
MAX_CHUNKS_PAR_PDF = 15    # Maximum de chunks traités par PDF
BASCULE_APRES = 8          # Basculer de modèle après N appels
SLEEP_ENTRE_APPELS = 2     # Secondes entre chaque appel API

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("data/generation.log", encoding="utf-8")
    ]
)
logger = logging.getLogger(__name__)

clients = {
    "model_1": ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=GROQ_API_KEY,
        temperature=0.3,
        max_tokens=1000
    ),
    "model_2": ChatGroq(
        model="llama-3.1-8b-instant",
        api_key=GROQ_API_KEY,
        temperature=0.3,
        max_tokens=1000
    )
}

# Compteur global pour la rotation
compteur_appels = 0


def get_client_actif():
    """
    Retourne le client actif selon le compteur de rotation.
    Alterne entre les 2 modèles toutes les BASCULE_APRES appels.
    """
    global compteur_appels
    index = (compteur_appels // BASCULE_APRES) % 2
    if index == 0:
        modele = "llama-3.3-70b-versatile"
        client = clients["model_1"]
    else:
        modele = "llama-3.1-8b-instant"
        client = clients["model_2"]
    compteur_appels += 1
    return client, modele



def charger_progression() -> dict:
    """Charge la progression sauvegardée pour reprendre si interruption."""
    if os.path.exists(FICHIER_PROGRESSION):
        with open(FICHIER_PROGRESSION, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"pdfs_termines": [], "total_qa": 0}


def sauvegarder_progression(pdfs_termines: list, total_qa: int):
    """Sauvegarde la progression pour reprendre en cas d'interruption."""
    with open(FICHIER_PROGRESSION, "w", encoding="utf-8") as f:
        json.dump({"pdfs_termines": pdfs_termines, "total_qa": total_qa}, f)


def extraire_texte_pdf(chemin_pdf: str) -> str:
    """Extrait le texte d'un fichier PDF page par page."""
    try:
        reader = pypdf.PdfReader(chemin_pdf)
        texte = ""
        for page in reader.pages:
            texte_page = page.extract_text()
            if texte_page:
                texte += texte_page + "\n"
        return texte.strip()
    except Exception as e:
        logger.error(f"Erreur extraction PDF {chemin_pdf} : {e}")
        return ""


def decouper_en_chunks(texte: str, taille: int = CHUNK_SIZE) -> list[str]:
    """
    Découpe le texte en chunks avec chevauchement de 20%.
    Évite de couper au milieu d'une phrase.
    """
    chunks = []
    mots = texte.split()
    chunk_actuel = []
    taille_actuelle = 0

    for mot in mots:
        chunk_actuel.append(mot)
        taille_actuelle += len(mot) + 1

        if taille_actuelle >= taille:
            chunks.append(" ".join(chunk_actuel))
            # Chevauchement de 20% pour ne pas perdre le contexte
            overlap = len(chunk_actuel) // 5
            chunk_actuel = chunk_actuel[-overlap:]
            taille_actuelle = sum(len(m) + 1 for m in chunk_actuel)

    # Ajouter le dernier chunk s'il reste du texte
    if chunk_actuel and taille_actuelle > 100:
        chunks.append(" ".join(chunk_actuel))

    return chunks


def extraire_temps_attente(message_erreur: str) -> int:
    """Extrait le temps d'attente en secondes depuis le message d'erreur Groq."""
    # Format : "Please try again in Xm Y.Zs"
    match_min = re.search(r"try again in (\d+)m", message_erreur)
    match_sec = re.search(r"(\d+)m(\d+\.\d+)s", message_erreur)

    if match_sec:
        minutes = int(match_sec.group(1))
        secondes = float(match_sec.group(2))
        return int(minutes * 60 + secondes) + 5  # +5s de marge
    elif match_min:
        return int(match_min.group(1)) * 60 + 30
    return 60  # Défaut : 60 secondes


def generer_qa_depuis_chunk(chunk: str, nom_fichier: str) -> list[dict]:
    """
    Génère des paires QA depuis un chunk de texte via Groq.
    Alterne automatiquement entre les 2 modèles.
    Gère les erreurs 429 avec attente et retry.
    """
    global compteur_appels 
    prompt = f"""Tu es un expert médical francophone. 
À partir du texte médical suivant, génère exactement {QUESTIONS_PAR_CHUNK} paires question-réponse en français.

Règles :
- Questions précises et cliniquement pertinentes
- Réponses complètes basées UNIQUEMENT sur le texte
- Varier les types : symptômes, traitements, diagnostics, épidémiologie, prévention
- Langue : français uniquement

Texte médical :
{chunk}

Réponds UNIQUEMENT avec un JSON valide, sans texte avant ou après :
[
  {{
    "question": "Question médicale précise ?",
    "answer": "Réponse détaillée et complète."
  }}
]"""

    max_tentatives = 3

    for tentative in range(max_tentatives):
        try:
            client_actif, modele_actif = get_client_actif()
            logger.info(f"    → [{modele_actif}] appel #{compteur_appels}")

            response = client_actif.invoke([HumanMessage(content=prompt)])
            contenu = response.content.strip()

            # Nettoyer les balises markdown si présentes
            if "```json" in contenu:
                contenu = contenu.split("```json")[1].split("```")[0].strip()
            elif "```" in contenu:
                contenu = contenu.split("```")[1].split("```")[0].strip()

            qa_pairs = json.loads(contenu)

            # Valider et nettoyer les résultats
            result = []
            for qa in qa_pairs:
                question = qa.get("question", "").strip()
                answer = qa.get("answer", "").strip()
                if question and answer and len(question) > 10 and len(answer) > 20:
                    result.append({
                        "question": question,
                        "answer": answer,
                        "context": chunk[:500],
                        "source": nom_fichier,
                        "modele": modele_actif
                    })
            return result

        except json.JSONDecodeError as e:
            logger.warning(f"    → Erreur JSON (tentative {tentative+1}) : {e}")
            time.sleep(2)
            continue

        except Exception as e:
            erreur_str = str(e)

            if "429" in erreur_str or "rate_limit" in erreur_str.lower():
                temps_attente = extraire_temps_attente(erreur_str)
                logger.warning(
                    f"    → Rate limit détecté — attente {temps_attente}s "
                    f"(tentative {tentative+1}/{max_tentatives})"
                )
                time.sleep(temps_attente)
                # Forcer la bascule vers l'autre modèle
                # Ajuster le compteur pour basculer de modèle
                compteur_appels = ((compteur_appels // BASCULE_APRES) + 1) * BASCULE_APRES
                continue
            else:
                logger.error(f"    → Erreur inattendue : {e}")
                return []

    logger.error(f"    → Échec après {max_tentatives} tentatives")
    return []


def formater_pour_finetuning(qa: dict) -> dict:
    """
    Formate une paire QA pour le fine-tuning.
    Format compatible avec Qwen2.5, Llama, Mistral (format messages).
    """
    return {
        "messages": [
            {
                "role": "system",
                "content": "Tu es un assistant médical expert en français. "
                           "Réponds aux questions médicales de manière précise, "
                           "claire et structurée, basée sur des données médicales fiables."
            },
            {
                "role": "user",
                "content": qa["question"]
            },
            {
                "role": "assistant",
                "content": qa["answer"]
            }
        ],
        "metadata": {
            "source": qa["source"],
            "modele_generation": qa.get("modele", "inconnu")
        }
    }


def main():
    logger.info("=" * 60)
    logger.info("GÉNÉRATION DATASET QA — FINE-TUNING MÉDICAL")
    logger.info(f"Rotation modèles : toutes les {BASCULE_APRES} requêtes")
    logger.info(f"Modèle 1 : llama-3.3-70b-versatile")
    logger.info(f"Modèle 2 : llama-3.1-8b-instant")
    logger.info("=" * 60)

    # Créer le dossier data si nécessaire
    os.makedirs(DATA_FOLDER, exist_ok=True)

    # Charger la progression (pour reprendre si interruption)
    progression = charger_progression()
    pdfs_termines = progression["pdfs_termines"]
    if pdfs_termines:
        logger.info(f"Reprise — {len(pdfs_termines)} PDFs déjà traités")

    # Récupérer tous les PDFs
    pdfs = sorted(Path(DATA_FOLDER).glob("*.pdf"))
    if not pdfs:
        logger.error(f"Aucun PDF trouvé dans {DATA_FOLDER}/")
        return

    logger.info(f"{len(pdfs)} PDFs trouvés")

    # Charger les QA déjà générés si le fichier existe
    tous_les_qa = []
    if os.path.exists(FICHIER_BRUT):
        with open(FICHIER_BRUT, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    tous_les_qa.append(json.loads(line))
        logger.info(f"{len(tous_les_qa)} QA déjà générés chargés")

    stats = {
        "pdfs_traites": len(pdfs_termines),
        "chunks_traites": 0,
        "qa_generes": len(tous_les_qa),
        "erreurs": 0,
        "appels_model_1": 0,
        "appels_model_2": 0
    }

    # Ouvrir le fichier brut en mode append
    fichier_brut_handle = open(FICHIER_BRUT, "a", encoding="utf-8")

    try:
        for i, pdf_path in enumerate(pdfs, 1):
            # Passer les PDFs déjà traités
            if pdf_path.name in pdfs_termines:
                logger.info(f"[{i}/{len(pdfs)}] {pdf_path.name} — déjà traité, ignoré")
                continue

            logger.info(f"\n[{i}/{len(pdfs)}] Traitement : {pdf_path.name}")

            # Extraction du texte
            texte = extraire_texte_pdf(str(pdf_path))
            if not texte or len(texte) < 200:
                logger.warning(f"  → Texte trop court ou vide, ignoré")
                stats["erreurs"] += 1
                pdfs_termines.append(pdf_path.name)
                continue

            logger.info(f"  → {len(texte):,} caractères extraits")

            # Découpage en chunks
            chunks = decouper_en_chunks(texte)
            chunks = chunks[:MAX_CHUNKS_PAR_PDF]
            logger.info(f"  → {len(chunks)} chunks à traiter")

            qa_pdf = []
            for j, chunk in enumerate(chunks, 1):
                logger.info(f"  → Chunk {j}/{len(chunks)}...")

                qa_pairs = generer_qa_depuis_chunk(chunk, pdf_path.name)
                qa_pdf.extend(qa_pairs)
                stats["chunks_traites"] += 1

                # Sauvegarder immédiatement dans le fichier brut
                for qa in qa_pairs:
                    fichier_brut_handle.write(
                        json.dumps(qa, ensure_ascii=False) + "\n"
                    )
                fichier_brut_handle.flush()

                # Statistiques par modèle
                for qa in qa_pairs:
                    if "70b" in qa.get("modele", ""):
                        stats["appels_model_1"] += 1
                    else:
                        stats["appels_model_2"] += 1

                # Pause entre appels
                time.sleep(SLEEP_ENTRE_APPELS)

            logger.info(f"  → {len(qa_pdf)} paires QA générées")
            tous_les_qa.extend(qa_pdf)
            stats["pdfs_traites"] += 1
            stats["qa_generes"] = len(tous_les_qa)

            # Sauvegarder la progression
            pdfs_termines.append(pdf_path.name)
            sauvegarder_progression(pdfs_termines, len(tous_les_qa))

    finally:
        fichier_brut_handle.close()

    # Générer le fichier final pour fine-tuning
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for qa in tous_les_qa:
            exemple = formater_pour_finetuning(qa)
            f.write(json.dumps(exemple, ensure_ascii=False) + "\n")

    # Afficher les statistiques finales
    logger.info("\n" + "=" * 60)
    logger.info("RÉSUMÉ FINAL")
    logger.info("=" * 60)
    logger.info(f"PDFs traités           : {stats['pdfs_traites']}/{len(pdfs)}")
    logger.info(f"Chunks traités         : {stats['chunks_traites']}")
    logger.info(f"Paires QA générées     : {stats['qa_generes']}")
    logger.info(f"Erreurs                : {stats['erreurs']}")
    logger.info(f"Appels llama-3.3-70b   : {stats['appels_model_1']}")
    logger.info(f"Appels llama-3.1-8b    : {stats['appels_model_2']}")
    logger.info(f"\nFichier fine-tuning    : {OUTPUT_FILE}")
    logger.info(f"Fichier brut           : {FICHIER_BRUT}")
    logger.info(f"Log complet            : data/generation.log")

    # Évaluation de la qualité du dataset
    logger.info("\n" + "=" * 60)
    logger.info("ÉVALUATION DU DATASET")
    logger.info("=" * 60)
    n = stats['qa_generes']
    if n < 500:
        logger.info(f"{n} QA — Insuffisant pour fine-tuning (min 500)")
    elif n < 1000:
        logger.info(f"✓  {n} QA — Acceptable pour fine-tuning")
    elif n < 3000:
        logger.info(f"✓✓ {n} QA — Bon dataset pour fine-tuning")
    else:
        logger.info(f"✓✓✓ {n} QA — Excellent dataset pour fine-tuning")

    # Aperçu des 3 premiers exemples
    if tous_les_qa:
        logger.info("\n=== APERÇU (3 premiers exemples) ===")
        for qa in tous_les_qa[:3]:
            logger.info(f"\nSource  : {qa['source']}")
            logger.info(f"Modèle  : {qa.get('modele', 'inconnu')}")
            logger.info(f"Q : {qa['question']}")
            logger.info(f"R : {qa['answer'][:200]}...")


if __name__ == "__main__":
    main()

2026-04-12 18:35:35,262 | ============================================================
2026-04-12 18:35:35,263 | GÉNÉRATION DATASET QA — FINE-TUNING MÉDICAL
2026-04-12 18:35:35,264 | Rotation modèles : toutes les 8 requêtes
2026-04-12 18:35:35,264 | Modèle 1 : llama-3.3-70b-versatile
2026-04-12 18:35:35,265 | Modèle 2 : llama-3.1-8b-instant
2026-04-12 18:35:35,266 | ============================================================
2026-04-12 18:35:35,268 | 18 PDFs trouvés
2026-04-12 18:35:35,301 | 210 QA déjà générés chargés
2026-04-12 18:35:35,302 | 
[1/18] Traitement : 241120_reco_392_reponse_rapide_codid-19_suivi_hta_maj.pdf
2026-04-12 18:35:35,828 |   → 19,870 caractères extraits
2026-04-12 18:35:35,830 |   → 15 chunks à traiter
2026-04-12 18:35:35,831 |   → Chunk 1/15...
2026-04-12 18:35:35,831 |     → [llama-3.3-70b-versatile] appel #1
2026-04-12 18:35:37,064 | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-12 18:35:39,067 |   → Chunk 2/15